# Analyze GotQuestions Categories

Explore `data/english/en_gotquestions.json` to identify categories useful for rebalancing the evaluation question set.

**Goals:**
- Doctrine Tier: at least 100 core, 100 secondary, 100 tertiary
- Flags: at least 100 applies_interfaith, 100 applies_evangelism

In [42]:
import json
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/english/en_gotquestions.json")
TAGS_PATH = Path("../data/english/en_question_tags.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    gotq_data = json.load(f)

with open(TAGS_PATH, "r", encoding="utf-8") as f:
    tags_data = json.load(f)

existing_questions = {tag["question"] for tag in tags_data["tags"]}
print(f"Loaded {len(gotq_data)} top-level categories")
print(f"Existing eval questions: {len(existing_questions)}")

Loaded 32 top-level categories
Existing eval questions: 500


In [35]:
# Extract all questions with their category path
all_questions = []

for cat in gotq_data:
    cat_name = cat["name"]
    for theme in cat.get("themes", []):
        theme_name = theme["name"]
        articles = theme.get("articles", {})
        if isinstance(articles, dict):
            for group_name, items in articles.items():
                for item in items:
                    all_questions.append({
                        "category": cat_name,
                        "theme": theme_name,
                        "group": group_name,
                        "question": item["name"],
                        "has_answer": bool(item.get("answer", "").strip()),
                        "already_in_eval": item["name"] in existing_questions,
                    })
        elif isinstance(articles, list):
            for item in articles:
                all_questions.append({
                    "category": cat_name,
                    "theme": theme_name,
                    "group": "(flat)",
                    "question": item["name"],
                    "has_answer": bool(item.get("answer", "").strip()),
                    "already_in_eval": item["name"] in existing_questions,
                })

df = pd.DataFrame(all_questions)
print(f"Total questions in gotquestions: {len(df)}")
print(f"Questions with answers: {df['has_answer'].sum()}")
print(f"Already in eval set: {df['already_in_eval'].sum()}")
print(f"Available (not in eval, has answer): {((~df['already_in_eval']) & df['has_answer']).sum()}")

Total questions in gotquestions: 10522
Questions with answers: 10522
Already in eval set: 610
Available (not in eval, has answer): 9912


In [36]:
# Top-level category question counts
cat_counts = df.groupby("category").agg(
    total=("question", "count"),
    available=("already_in_eval", lambda x: (~x).sum()),
).sort_values("total", ascending=False)

print("\nCategory Question Counts:")
print(cat_counts.to_string())


Category Question Counts:
                                        total  available
category                                                
Questions about the Books of the Bible   3259       3128
Questions about People in the Bible       494        490
Questions about Worldview                 442        418
Questions about Life                      436        397
Questions about the Bible                 434        413
Questions about Christian History         331        328
Questions about the Church                327        285
Questions about Sin                       324        306
Topical Bible Questions                   319        310
Questions about Cults and Religions       311        279
Questions about Spiritual Life            297        278
Questions about Jesus Christ              259        242
Questions about God                       256        225
Questions about Christianity              231        222
Questions about False Beliefs             218        204
Ques

In [37]:
# Categories likely to yield CORE doctrine questions
core_categories = [
    "Questions about God",
    "Questions about Jesus Christ",
    "Questions about the Holy Spirit",
    "Questions about Salvation",
    "Questions about Theology",
]

print("\n=== Categories likely to yield CORE doctrine questions ===")
for cat_name in core_categories:
    cat_df = df[(df["category"] == cat_name) & (~df["already_in_eval"]) & df["has_answer"]]
    print(f"\n{cat_name}: {len(cat_df)} available questions")
    themes = cat_df.groupby("theme")["question"].count().sort_values(ascending=False)
    for theme, count in themes.items():
        print(f"  {theme}: {count}")


=== Categories likely to yield CORE doctrine questions ===

Questions about God: 225 available questions
  The Character of God: 53
  The Nature of God: 44
  God and Evil: 35
  The Identity of God: 34
  Our Response to God: 30
  God's Relationship to Us: 29

Questions about Jesus Christ: 242 available questions
  The Cross and the Empty Tomb: 46
  The Ministry of Jesus: 41
  The Life of Jesus: 37
  The Relationships of Jesus: 34
  The Identity of Jesus: 32
  The Nature of Jesus: 27
  The Person of Jesus: 25

Questions about the Holy Spirit: 68 available questions
  The Gifts of the Holy Spirit: 26
  The Identity of the Holy Spirit: 21
  The Work of the Holy Spirit: 21

Questions about Salvation: 170 available questions
  Theology of Salvation: 44
  The Gospel: 36
  The Truths of Salvation: 35
  Salvation Clarified: 34
  Who can be Saved: 21

Questions about Theology: 190 available questions
  Systematic Theology: 38
  God and Humanity: 36
  Calvinism and Arminianism: 35
  Theology of 

In [38]:
# Categories likely to yield SECONDARY doctrine questions
secondary_categories = [
    "Questions about the Church",
    "Questions about Theology",
    "Questions about the End Times",  # some secondary (atonement theories), some tertiary
]

print("\n=== Categories likely to yield SECONDARY doctrine questions ===")
for cat_name in secondary_categories:
    cat_df = df[(df["category"] == cat_name) & (~df["already_in_eval"]) & df["has_answer"]]
    print(f"\n{cat_name}: {len(cat_df)} available questions")
    themes = cat_df.groupby("theme")["question"].count().sort_values(ascending=False)
    for theme, count in themes.items():
        print(f"  {theme}: {count}")


=== Categories likely to yield SECONDARY doctrine questions ===

Questions about the Church: 285 available questions
  Rites and Ordinances: 48
  Church Worship: 41
  Church Ministry: 39
  Church Organization: 38
  The Body of Christ: 32
  Church Culture: 31
  Church Leadership: 31
  Church Identity: 25

Questions about Theology: 190 available questions
  Systematic Theology: 38
  God and Humanity: 36
  Calvinism and Arminianism: 35
  Theology of the Trinity: 31
  Dispensationalism and Covenantalism: 26
  Basics: 24

Questions about the End Times: 171 available questions
  Events: 54
  People and Nations: 44
  Keep Watch: 43
  Eschatology: 30


In [39]:
# Categories likely to yield TERTIARY doctrine questions
tertiary_categories = [
    "Questions about the End Times",  # eschatology is tertiary
    "Questions about Family",
    "Questions about Life",
    "Questions about Relationships",
]

print("\n=== Categories likely to yield TERTIARY doctrine questions ===")
for cat_name in tertiary_categories:
    cat_df = df[(df["category"] == cat_name) & (~df["already_in_eval"]) & df["has_answer"]]
    print(f"\n{cat_name}: {len(cat_df)} available questions")
    themes = cat_df.groupby("theme")["question"].count().sort_values(ascending=False)
    for theme, count in themes.items():
        print(f"  {theme}: {count}")


=== Categories likely to yield TERTIARY doctrine questions ===

Questions about the End Times: 171 available questions
  Events: 54
  People and Nations: 44
  Keep Watch: 43
  Eschatology: 30

Questions about Family: 104 available questions
  Raising Children: 33
  Having Children: 27
  Family Relationships: 26
  Family Basics: 18

Questions about Life: 397 available questions
  The Christian and Holidays: 72
  Personal Interaction: 50
  The Christian and Money: 50
  Christian Character: 48
  Life in Practice: 46
  Growth in Life: 37
  Christian Evangelism: 36
  The Christian and Culture: 33
  Life Essentials: 25

Questions about Relationships: 185 available questions
  Getting Married: 39
  Marital Unity: 37
  Marriage Hardships: 34
  Dating: 29
  Friendship: 23
  Marriage Basics: 23


In [40]:
# Categories likely to yield INTERFAITH / EVANGELISM questions
interfaith_categories = [
    "Questions about Apologetics",
    "Questions about Cults and Religions",
    "Questions about Worldview",
    "Questions about Islam",
    "Questions about Catholicism",
]

print("\n=== Categories likely to yield INTERFAITH + EVANGELISM questions ===")
for cat_name in interfaith_categories:
    cat_df = df[(df["category"] == cat_name) & (~df["already_in_eval"]) & df["has_answer"]]
    print(f"\n{cat_name}: {len(cat_df)} available questions")
    themes = cat_df.groupby("theme")["question"].count().sort_values(ascending=False)
    for theme, count in themes.items():
        print(f"  {theme}: {count}")


=== Categories likely to yield INTERFAITH + EVANGELISM questions ===

Questions about Apologetics: 78 available questions
  Apologetics Basics: 29
  Apologetics in Christianity: 27
  Apologetics and Unbelievers: 22

Questions about Cults and Religions: 279 available questions
  Major World Religions: 66
  Regional Religions: 51
  Minor World Religions: 46
  Cults and Religions Basics: 44
  Religious Terms and Practices: 42
  The Occult: 30

Questions about Worldview: 418 available questions
  Worldview on Social Issues: 81
  Government and Worldview: 60
  Worldview and Philosophy: 57
  Worldview and Ethics: 51
  Entertainment and Worldview: 48
  Worldview Application: 43
  Worldview Essentials: 42
  Conflict and Worldview: 36

Questions about Islam: 53 available questions
  Islamic beliefs: 26
  Islamic Sects and Writings: 19
  Bible Studies for Muslims: 8

Questions about Catholicism: 150 available questions
  Catholic Beliefs: 36
  Catholicism and Tradition: 33
  Catholicism Basics:

In [41]:
# Show current tag distribution for reference
print("\n=== Current Eval Set Distribution ===")
tags = tags_data["tags"]

from collections import Counter
tier_counts = Counter(t["doctrine_tier"] for t in tags)
type_counts = Counter(t["question_type"] for t in tags)

flag_names = [
    "applies_core_doctrine", "applies_secondary_doctrine", "applies_tertiary_handling",
    "applies_pastoral", "applies_interfaith", "applies_evangelism",
]
flag_counts = {f: sum(1 for t in tags if t.get(f)) for f in flag_names}

print("\nDoctrine Tier:")
for tier, count in sorted(tier_counts.items()):
    deficit = max(0, 100 - count)
    print(f"  {tier}: {count}" + (f" (NEED {deficit} more)" if deficit > 0 else " (OK)"))

print("\nFlag Counts:")
for flag, count in flag_counts.items():
    deficit = max(0, 100 - count)
    print(f"  {flag}: {count}" + (f" (NEED {deficit} more)" if deficit > 0 else " (OK)"))

print("\nQuestion Type:")
for qtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {qtype}: {count}")


=== Current Eval Set Distribution ===

Doctrine Tier:
  core: 138 (OK)
  not_directly_doctrinal: 160 (OK)
  secondary: 100 (OK)
  tertiary: 102 (OK)

Flag Counts:
  applies_core_doctrine: 156 (OK)
  applies_secondary_doctrine: 111 (OK)
  applies_tertiary_handling: 126 (OK)
  applies_pastoral: 127 (OK)
  applies_interfaith: 100 (OK)
  applies_evangelism: 100 (OK)

Question Type:
  doctrinal: 236
  factual_historical: 63
  pastoral: 63
  practical_ethical: 54
  apologetic_interfaith: 45
  bible_survey: 20
  methodological: 13
  comparative_religion: 6


# New!

Doctrine Tier Distribution:
* core: 138 (27.6%)
* not_directly_doctrinal: 160 (32.0%)
* secondary: 100 (20.0%)
* tertiary: 102 (20.4%)

Question Type Distribution:
* apologetic_interfaith: 45 (9.0%)
* bible_survey: 20 (4.0%)
* comparative_religion: 6 (1.2%)
* doctrinal: 236 (47.2%)
* factual_historical: 63 (12.6%)
* methodological: 13 (2.6%)
* pastoral: 63 (12.6%)
* practical_ethical: 54 (10.8%)

Flag Counts (true):
* applies_core_doctrine: 156 (31.2%)
* applies_secondary_doctrine: 111 (22.2%)
* applies_tertiary_handling: 126 (25.2%)
* applies_pastoral: 127 (25.4%)
* applies_interfaith: 100 (20.0%)
* applies_evangelism: 100 (20.0%)

Questions with all flags false: 29 (5.8%)